In [1]:
import pandas as pd

bankdf = pd.read_csv("Dataset/BankData.csv")
bankdf['Min_Income_Required'] = bankdf['Min_Income_Required'].str.replace(',','').astype(int)
bankdf['Limit'] = bankdf['Limit'].str.replace(',','').astype(int)
bankdf['Lounge_access'] = bankdf['Lounge_access'].fillna(0).astype(int)
# bankdf.dtypes

In [2]:
customerdf = pd.read_csv("Dataset/Customer Spends.csv")
# customerdf['Annual_Income'].max()
# customerdf.columns
# customerdf.head(1)

In [92]:
# bankdf.columns

In [93]:
'''
Identify which credit card offers the highest cashback on grocery 
purchases (for example, from Zepto, BigBasket, etc.)
based on the available percentage in the dataset.
'''

bankdf[(bankdf['Grocery_Cashback'] == max(bankdf['Grocery_Cashback']))
& (bankdf['CardType'] == 'Credit')][['Bank_Name','Card_Name','Grocery_Cashback']]

,Bank_Name,Card_Name,Grocery_Cashback
26,Bank of Baroda,HPCL Credit Card,10.0
27,Bank of Baroda,IRCTC Credit Card,10.0
28,Bank of Baroda,Prime Credit Card,10.0


In [94]:
'''
 Filter out credit cards that do not have an annual fee and show 
 them to customers who prefer no upfront costs. 
'''

bankdf[(bankdf['Annual_Fee'] == 0) & 
(bankdf['CardType'] == "Credit")][['Bank_Name','Card_Name']]

,Bank_Name,Card_Name
28,Bank of Baroda,Prime Credit Card
29,Canara Bank,VISA Classic Credit Card
30,Canara Bank,MasterCard Standard
31,Canara Bank,RuPay Platinum Credit Card
63,Federal Bank,Visa Celesta
64,Federal Bank,Visa Imperio
65,Federal Bank,Visa Signet
66,Federal Bank,Mastercard Celesta
67,Federal Bank,Mastercard Imperio
68,Federal Bank,Mastercard Signet


In [95]:
'''
 Identify credit cards that are available to a customer
 based on their income level.
'''

creditCards = bankdf[bankdf['CardType'] == "Credit"]
merged = pd.merge(customerdf, creditCards, how = "cross")
merged[merged['Min_Income_Required'] <= merged['Annual_Income']][['Customer_ID','Name',
                            'Annual_Income','Min_Income_Required','Bank_Name','Card_Name']]

,Customer_ID,Name,Annual_Income,Min_Income_Required,Bank_Name,Card_Name
1,CUS_001,Aarav Sharma,250000,240000,SBI,SimplyCLICK SBI Card
2,CUS_001,Aarav Sharma,250000,240000,SBI,SimplySAVE SBI Card
5,CUS_001,Aarav Sharma,250000,240000,SBI,IRCTC SBI Card Platinum
8,CUS_001,Aarav Sharma,250000,15000,Bank of Baroda,Prime Credit Card
9,CUS_001,Aarav Sharma,250000,150000,Canara Bank,VISA Classic Credit Card
...,...,...,...,...,...,...
2810,CUS_101,Lokesh G,180000,150000,Canara Bank,MasterCard Standard
2836,CUS_102,Ashwin K,210000,15000,Bank of Baroda,Prime Credit Card
2837,CUS_102,Ashwin K,210000,150000,Canara Bank,VISA Classic Credit Card
2838,CUS_102,Ashwin K,210000,150000,Canara Bank,MasterCard Standard


In [96]:
'''
Calculate the total cashback a customer would receive based 
on their spending across groceries, travel, and dining categories. 
'''

merged = pd.merge(customerdf, bankdf[bankdf['CardType'] == "Credit"], how="cross")
eligible = merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()
eligible['Total_Monthly_Cashback'] = (
    (eligible['Grocery_Spend'] * eligible['Grocery_Cashback'] / 100) +
    (eligible['Travel_Spend'] * eligible['Travel_Cashback'] / 100) +
    (eligible['Dining_Spend'] * eligible['Dining_Cashback'] / 100)
)
eligible[['Name', 'Bank_Name', 'Card_Name', 'Total_Monthly_Cashback']]

,Name,Bank_Name,Card_Name,Total_Monthly_Cashback
1,Aarav Sharma,SBI,SimplyCLICK SBI Card,0.0
2,Aarav Sharma,SBI,SimplySAVE SBI Card,0.0
5,Aarav Sharma,SBI,IRCTC SBI Card Platinum,200.0
8,Aarav Sharma,Bank of Baroda,Prime Credit Card,800.0
9,Aarav Sharma,Canara Bank,VISA Classic Credit Card,100.0
...,...,...,...,...
2810,Lokesh G,Canara Bank,MasterCard Standard,70.0
2836,Ashwin K,Bank of Baroda,Prime Credit Card,750.0
2837,Ashwin K,Canara Bank,VISA Classic Credit Card,90.0
2838,Ashwin K,Canara Bank,MasterCard Standard,90.0


In [103]:
'''
Find the credit cards that offer the highest cashback 
on dining (e.g., through Swiggy, Zomato). 
'''

creditCards.sort_values(by = 'Dining_Cashback', ascending = False).head(10)[['Bank_Name', 'Card_Name', 'Dining_Cashback']]

,Bank_Name,Card_Name,Dining_Cashback
18,SBI,Cashback SBI Card,5.0
76,Kotak Mahindra Bank,PVR INOX Kotak Credit Card,5.0
72,Kotak Mahindra Bank,Kotak Cashback+ Credit Card,5.0
31,Canara Bank,RuPay Platinum Credit Card,2.0
32,Canara Bank,Canara Gold Credit Card,2.0
29,Canara Bank,VISA Classic Credit Card,1.0
30,Canara Bank,MasterCard Standard,1.0
66,Federal Bank,Mastercard Celesta,0.0
75,Kotak Mahindra Bank,Kotak Air Credit Card,0.0
74,Kotak Mahindra Bank,Zen Signature Credit Card,0.0


In [109]:
'''
Compare the annual fee of credit cards with the 
total cashback value a customer would get.
'''

merged = pd.merge(customerdf, bankdf[bankdf['CardType'] == "Credit"], how="cross")
eligible = merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()

eligible['Annual_Grocery_CB'] = (eligible['Grocery_Spend'] * 12) * (eligible['Grocery_Cashback'] / 100)
eligible['Annual_Travel_CB'] = (eligible['Travel_Spend'] * 12) * (eligible['Travel_Cashback'] / 100)
eligible['Annual_Dining_CB'] = (eligible['Dining_Spend'] * 12) * (eligible['Dining_Cashback'] / 100)
eligible['Total_Annual_Cashback'] = eligible['Annual_Grocery_CB'] + eligible['Annual_Travel_CB'] + eligible['Annual_Dining_CB']

eligible['Net_Annual_Benefit'] = eligible['Total_Annual_Cashback'] - eligible['Annual_Fee']

eligible.sort_values(by=['Name', 'Net_Annual_Benefit'], ascending=[True, False])[['Customer_ID','Name','Bank_Name','Card_Name','Annual_Income',
                                        'Total_Annual_Cashback','Net_Annual_Benefit']]

,Customer_ID,Name,Bank_Name,Card_Name,Annual_Income,Total_Annual_Cashback,Net_Annual_Benefit
2304,CUS_083,A R Rahman,Bank of Baroda,Prime Credit Card,150000,10200.0,10200.0
2305,CUS_083,A R Rahman,Canara Bank,VISA Classic Credit Card,150000,1140.0,1140.0
2306,CUS_083,A R Rahman,Canara Bank,MasterCard Standard,150000,1140.0,1140.0
8,CUS_001,Aarav Sharma,Bank of Baroda,Prime Credit Card,250000,9600.0,9600.0
11,CUS_001,Aarav Sharma,Canara Bank,RuPay Platinum Credit Card,250000,2400.0,2400.0
...,...,...,...,...,...,...,...
2026,CUS_073,Vikram,Canara Bank,MasterCard Standard,160000,1200.0,1200.0
64,CUS_003,Vikram Singh,Bank of Baroda,Prime Credit Card,180000,6900.0,6900.0
65,CUS_003,Vikram Singh,Canara Bank,VISA Classic Credit Card,180000,840.0,840.0
66,CUS_003,Vikram Singh,Canara Bank,MasterCard Standard,180000,840.0,840.0


In [122]:
'''
 Identify cards that offer a monthly spending limit suitable for the customer's financial needs. 
'''

merged = pd.merge(customerdf, bankdf[bankdf['CardType']=="Credit"], how="cross")
eligible= merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()
suitable_cards = eligible[eligible['Limit'] >= eligible['Monthly_Spend']].copy()
suitable_cards['Limit_Buffer'] = suitable_cards['Limit'] - suitable_cards['Monthly_Spend']
suitable_cards.sort_values(by='Limit_Buffer', ascending=False).head(20)[['Name', 'Monthly_Spend', 'Bank_Name', 'Card_Name', 'Limit', 'Limit_Buffer']]

,Name,Monthly_Spend,Bank_Name,Card_Name,Limit,Limit_Buffer
2654,Gautham M,22000,Kotak Mahindra Bank,Kotak Cashback+ Credit Card,400000,378000
2738,Vignesh S,22000,Kotak Mahindra Bank,Kotak Cashback+ Credit Card,400000,378000
2682,Nelson D,25000,Kotak Mahindra Bank,Kotak Cashback+ Credit Card,400000,375000
23,Aarav Sharma,15000,Kotak Mahindra Bank,IndianOil Kotak Credit Card,300000,285000
27,Aarav Sharma,15000,Kotak Mahindra Bank,Kotak UPI RuPay Credit Card,300000,285000
2799,H Vinoth,18000,Kotak Mahindra Bank,Kotak UPI RuPay Credit Card,300000,282000
2795,H Vinoth,18000,Kotak Mahindra Bank,IndianOil Kotak Credit Card,300000,282000
2711,Pradeep R,20000,Kotak Mahindra Bank,IndianOil Kotak Credit Card,300000,280000
2715,Pradeep R,20000,Kotak Mahindra Bank,Kotak UPI RuPay Credit Card,300000,280000
2739,Vignesh S,22000,Kotak Mahindra Bank,IndianOil Kotak Credit Card,300000,278000


In [132]:
'''
Find credit cards that provide the highest reward points for every rupee spent
'''

merged = pd.merge(customerdf, bankdf[bankdf['CardType']=="Credit"], how="cross")
eligible= merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()
eligible['Monthly_Point_Value_INR'] = eligible['Monthly_Spend'] * eligible['Reward_Points']
eligible[['Name', 'Monthly_Spend', 'Bank_Name', 'Card_Name', 'Monthly_Point_Value_INR']].sort_values(
    by='Monthly_Point_Value_INR', ascending=False).head(15)

,Name,Monthly_Spend,Bank_Name,Card_Name,Monthly_Point_Value_INR
2681,Nelson D,25000,Kotak Mahindra Bank,League Platinum Credit Card,125000.0
2671,Nelson D,25000,Canara Bank,RuPay Platinum Credit Card,125000.0
2672,Nelson D,25000,Canara Bank,Canara Gold Credit Card,125000.0
2653,Gautham M,22000,Kotak Mahindra Bank,League Platinum Credit Card,110000.0
2727,Vignesh S,22000,Canara Bank,RuPay Platinum Credit Card,110000.0
2728,Vignesh S,22000,Canara Bank,Canara Gold Credit Card,110000.0
2644,Gautham M,22000,Canara Bank,Canara Gold Credit Card,110000.0
2643,Gautham M,22000,Canara Bank,RuPay Platinum Credit Card,110000.0
2737,Vignesh S,22000,Kotak Mahindra Bank,League Platinum Credit Card,110000.0
2699,Pradeep R,20000,Canara Bank,RuPay Platinum Credit Card,100000.0


In [35]:
'''
Identify credit cards that offer lounge access and match the customer's preferences for travel benefits.
'''
merged = pd.merge(customerdf, bankdf[bankdf['CardType']=="Credit"], how="cross")
eligible= merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()
eligible[(eligible['Travel_Spend']//357 >= eligible['Lounge_access']) & (eligible['Lounge_access'] > 0)][['Customer_ID',
                            'Name','Travel_Spend','Bank_Name','Card_Name','Lounge_access']].sort_values(by = 'Lounge_access', ascending = False)

,Customer_ID,Name,Travel_Spend,Bank_Name,Card_Name,Lounge_access
2728,CUS_098,Vignesh S,4000,Canara Bank,Canara Gold Credit Card,5
2727,CUS_098,Vignesh S,4000,Canara Bank,RuPay Platinum Credit Card,5
11,CUS_001,Aarav Sharma,2000,Canara Bank,RuPay Platinum Credit Card,5
2644,CUS_095,Gautham M,5000,Canara Bank,Canara Gold Credit Card,5
2671,CUS_096,Nelson D,5000,Canara Bank,RuPay Platinum Credit Card,5
...,...,...,...,...,...,...
1185,CUS_043,Rajkummar R,1000,Canara Bank,VISA Classic Credit Card,2
2838,CUS_102,Ashwin K,1000,Canara Bank,MasterCard Standard,2
2735,CUS_098,Vignesh S,4000,Federal Bank,Rupay Signet,1
2651,CUS_095,Gautham M,5000,Federal Bank,Rupay Signet,1


In [9]:
'''
Provide personalized credit card recommendations based on a customer’s profile, which includes income, 
spending habits (grocery, dining, travel), and reward preferences.
'''

merged = pd.merge(customerdf, bankdf[bankdf['CardType'] == "Credit"], how="cross")
eligible = merged[merged['Annual_Income'] >= merged['Min_Income_Required']].copy()
eligible = eligible[eligible['Limit'] >= (eligible['Monthly_Spend'] * 2)]
pref_map = {
    'Grocery': 'Grocery_Cashback',
    'Dining': 'Dining_Cashback',
    'Travel': 'Travel_Cashback'
}
def calculate_score(row):
    pref_col = pref_map.get(row['Preferred_Category'], 'Grocery_Cashback')
    # Weight the preferred category higher
    score = (row[pref_col] * 3) + row['Reward_Points']
    # Penalize high fees unless they are offset by points
    score -= (row['Annual_Fee'] / 500) 
    return score

eligible['Match_Score'] = eligible.apply(calculate_score, axis=1)

top_recommendations = eligible.sort_values(['Name', 'Match_Score'], ascending=[True, False]).groupby('Name').head(1)

top_recommendations[['Name', 'Preferred_Category', 'Bank_Name', 'Card_Name', 'Match_Score']]

,Name,Preferred_Category,Bank_Name,Card_Name,Match_Score
2304,A R Rahman,Grocery,Bank of Baroda,Prime Credit Card,32.0
8,Aarav Sharma,Grocery,Bank of Baroda,Prime Credit Card,32.0
1492,Aditi Rao,Grocery,Bank of Baroda,Prime Credit Card,32.0
456,Aditya Bose,Grocery,Bank of Baroda,Prime Credit Card,32.0
2052,Ajith K,Grocery,Bank of Baroda,Prime Credit Card,32.0
...,...,...,...,...,...
1688,Vijay D,Grocery,Bank of Baroda,Prime Credit Card,32.0
1408,Vijay V,Grocery,Bank of Baroda,Prime Credit Card,32.0
2024,Vikram,Grocery,Bank of Baroda,Prime Credit Card,32.0
64,Vikram Singh,Grocery,Bank of Baroda,Prime Credit Card,32.0
